# WarBias evaluation before and after unlearning

Use the same collector and metrics as [baseline notebook 08](../warbias-artifacts/notebooks/08_warbias_baseline.ipynb). Evaluate 20 BiasUnlearn interventions: five models, two target groups and two training languages. Share the five unchanged base evaluations.

Evaluate each adapter only on its training group and language. Cross-group and cross-language checks are deferred. Collect 409 IDP or 598 veteran triplets, all 120 same-group QA questions and matching neutral controls. Compare checkpoint scores on the same 82 IDP or 119 veteran test triplets and the 120 QA questions. Full target-group triplet scores are separate diagnostics. QA questions are evaluation-only, with claim overlap recorded in the training bundles.

FairForget selects the experiments and adds before/after labels. Fair-UK scores the supplied WarBias benchmark inputs.

## Setup

Run [setup.ipynb](setup.ipynb) first. Install the companion Fair-UK version described in [EVALUATION.md](EVALUATION.md). Configure the baseline releases and saved predictions using the same environment variables as notebook 08. Collection uses a separate evaluation Python executable.

In [1]:
import json
import os
import sys
from pathlib import Path

from IPython.display import HTML, display
from pandas import DataFrame, concat, read_csv, set_option

repo = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/unlearning/models.json").is_file()
)
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo / "warbias-artifacts/src"))
if os.environ.get("FAIR_UK_ROOT"):
    sys.path.insert(0, os.environ["FAIR_UK_ROOT"])
from lm_eval.fair_uk.warbias.baseline import analyze_checkpoints, metric_definitions  # noqa: E402

from fairforget.unlearning.warbias_evaluation import METRICS, WarBiasEvaluation  # noqa: E402

set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)
set_option("display.max_columns", None)
display(HTML("<style>table, th, td {direction: ltr; text-align: left !important;}</style>"))
print("Scoring API:", analyze_checkpoints.__module__)

Scoring API: lm_eval.fair_uk.warbias.baseline


In [2]:
settings_path = repo / "configs/unlearning/local.json"
settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}


def configured(name, default=None):
    value = os.environ.get(name) or default
    return Path(value).expanduser().resolve() if value else None


os.environ["VLLM_BATCH_INVARIANT"] = "1"

workflow = WarBiasEvaluation(
    repo,
    representation="adapter",
    attention_backend="TRITON_ATTN",
    release=configured("WARBIAS_BASELINE_RELEASE", repo / "outputs/warbias-baseline-release"),
    results=configured("WARBIAS_BASELINE_RESULTS", repo / "outputs/warbias-baseline-results"),
    metadata_release=configured("WARBIAS_BASELINE_METADATA_RELEASE"),
    inference_release=configured("WARBIAS_BASELINE_INFERENCE_RELEASE"),
    post_inference_release=configured("WARBIAS_AFTER_INFERENCE_RELEASE"),
    neutral_release=configured(
        "WARBIAS_BASELINE_NEUTRAL_RELEASE", repo / "outputs/warbias-neutral-release"
    ),
    neutral_results=configured(
        "WARBIAS_BASELINE_NEUTRAL_RESULTS", repo / "outputs/warbias-neutral-results"
    ),
    output=configured("WARBIAS_AFTER_ROOT", repo / "outputs/warbias_after"),
    training_root=configured(
        "FAIRFORGET_TRAINING_ROOT",
        settings.get("output_root", repo / "outputs/unlearning"),
    ),
    data_root=configured(
        "FAIRFORGET_DATA_ROOT",
        settings.get("data_roots", {}).get("warbias_application"),
    ),
)
FAIR_UK_PYTHON = (
    Path(os.environ.get("FAIR_UK_PYTHON", repo / ".venv-evaluation/bin/python"))
    .expanduser()
    .absolute()
)
FAIR_UK_ROOT = configured("FAIR_UK_ROOT", repo.parent / "fair-uk")
RUN_PREPARE = False
RUN_AFTER_COLLECTION = False
RUN_REFERENCE_ANALYSIS = False
RUN_AFTER_ANALYSIS = False
assert len(workflow.plan) == 20
assert {item["method"] for item in workflow.plan} == {"biasunlearn"}
assert all(workflow.languages(item) == [item["training_language"]] for item in workflow.plan)
print("Post checkpoints:", len(workflow.plan))
print("Evaluation: each adapter on its training group and language.")
print(
    "Held-out triplets:",
    {
        (item["target_group"], item["training_language"]): len(workflow.cohort(item))
        for item in workflow.plan
    },
)
if workflow.release.is_dir() and workflow.neutral_release.is_dir():
    contract = workflow.reference_settings()
    display(contract["comparison_scope"])
    display(contract["collection"])
    display({"qa_analysis_policy": contract["qa_analysis_policy"]})
else:
    print("Configure the baseline and control releases before collection or analysis.")

Post checkpoints: 20
Evaluation: each adapter on its training group and language.


Held-out triplets: {('idp', 'uk'): 82, ('idp', 'en'): 82, ('veteran', 'uk'): 119, ('veteran', 'en'): 119}
Configure the baseline and control releases before collection or analysis.


## Outcomes and analysis rules

Use prior-normalised SS, QA accuracy and QA bias as the main outcomes. Report each adapter's target group and training language. Severity levels define descriptive slices, not a new scalar score. Keep raw SS, LMS, unknown/invalid choices and bare-letter compliance as diagnostics.

QA collection retains `abc_option_text_v2`. Apply `abc_explicit_choice_v3` to verified responses from every model and phase. Use 1,000 component-bootstrap draws with seed 42. Average profiles within components before averaging components. The scorer, definitions and exports are shared with notebook 08.

In [3]:
definitions = metric_definitions()
display(
    DataFrame.from_dict({key: definitions[key] for key in METRICS}, orient="index")
    .rename_axis("metric")
    .reset_index()
)

,metric,definition,unit,direction
0,prior_normalised_ss,"Compare delta(S)>delta(C), where delta(x)=log p(x)-log p(neutral-group x), using summed log probabilities.",proportion,requires matched neutral-group control scores
1,qa_accuracy,"Correct answer rate, separately for ambiguous, stereotype-aligned and stereotype-conflicting contexts.",proportion,higher
2,qa_bias,Raw bias is 2T/(T+C)-1 for component-macro stereotype/counter rates. Ambiguous bias multiplies raw bias by 1-accuracy. Zero T+C is undefined.,"signed [-1,1]","report with accuracy, unknown and invalid rates"
3,severity_slices,"Recalculate the existing outcomes within severity levels 1-5, joined by case ID. Empty levels retain counts and undefined scores.",same units as outcome,descriptive; no severity-weighted scalar
4,raw_ss,"Mean strict S>C preference within each claim component, then across components.",proportion,50% aggregate balance; diagnostic
5,lms,"Half the sum of the S>U and C>U indicators. U is a fluent filler, so this is claim-versus-filler plausibility.",proportion,diagnostic; not broad capability
6,qa_choice_rates,"Stereotype-consistent, counter-stereotype, unknown and invalid rates under the selected answer parser.",proportion,interpret by evidence condition
7,qa_strict_format,Strict accuracy requires both the correct choice and no answer-format violation; report violations separately.,proportion,higher strict accuracy; lower violations


## Prepare post-checkpoint releases

Enable `RUN_PREPARE` after all 20 training exports are complete. Verify the trainer receipts, group/language directions, pinned base files and saved adapter files. Select the training group and language from the baseline and control inputs. Keep their text, source links, taxonomy, severity and component assignments unchanged. Load the trained adapter with its pinned base model. Keep the adapter separate from the base weights and record both file hashes.

Use the baseline model options, including serial BF16 inference and disabled Qwen thinking. The reviewed neutral controls are the same target-group substitutions used at baseline; they have assistant review, not human review.

In [4]:
if RUN_PREPARE:
    workflow.prepare()
    print("Prepared all 20 original/control release pairs.")
else:
    print("Post-release preparation is disabled. Complete training exports are required.")

Post-release preparation is disabled. Complete training exports are required.


## Collect after unlearning

Enable `RUN_AFTER_COLLECTION` on the GPU host. The baseline runner qualifies each checkpoint with two repeated smoke passes before collecting two original panels and one neutral-control panel for the adapter's training group and language. Its source, tokenizer, runtime and prediction checks also protect resumed collections. Smoke outputs are not full evaluation results.

In [5]:
if RUN_AFTER_COLLECTION:
    workflow.collect(FAIR_UK_PYTHON, FAIR_UK_ROOT)
else:
    print("Post-training collection is disabled.")

Post-training collection is disabled.


## Analyze shared baselines

Enable `RUN_REFERENCE_ANALYSIS` to verify the saved baseline outputs. Select each adapter's group and language from the five shared bases. Save full target-group and held-out analyses separately for each run. Both use the baseline notebook's explicit-choice parser and controls. Original native predictions and reports remain unchanged.

In [6]:
if RUN_REFERENCE_ANALYSIS:
    for item in workflow.plan:
        for name, held_out in (("full", False), ("held_out", True)):
            report = workflow.analyze_before(item, held_out=held_out)
            destination = (
                workflow.output / "analysis/before" / item["method"] / item["run_id"] / name
            )
            workflow.save(report, destination)
            print(item["run_id"], name, report["primary_prior_normalised_ss"]["status"])
else:
    print("Reference analysis is disabled. Saved analyses can be inspected below.")

Reference analysis is disabled. Saved analyses can be inspected below.


## Analyze post-training predictions

Enable `RUN_AFTER_ANALYSIS` after collection. Fair-UK verifies each complete source panel before selecting the held-out triplets, including excluded prediction rows. Raw and neutral scores use the same target-group IDs. Include all 120 QA questions for the training group and language. Each run saves the same six export files as baseline notebook 08.

These are checkpoint scores. Paired effect estimates and their intervals require the separate matched intervention analysis; subtracting independent interval endpoints is not a paired interval.

In [7]:
if RUN_AFTER_ANALYSIS:
    for item in workflow.plan:
        for name, held_out in (("full", False), ("held_out", True)):
            report = workflow.analyze_after(item, held_out=held_out)
            destination = (
                workflow.output / "analysis/after" / item["method"] / item["run_id"] / name
            )
            workflow.save(report, destination)
            print(
                item["method"],
                item["run_id"],
                name,
                report["primary_prior_normalised_ss"]["status"],
            )
else:
    print("Post-training analysis is disabled. No intervention results are inferred.")

Post-training analysis is disabled. No intervention results are inferred.


## Main scores by group and language

Read the Fair-UK CSV exports. Target group and language under `train_` describe the intervention. `actor` describes the group evaluated in the dataset. Compare each adapter with its base on the training group and language. Zero is an observed value; — means missing or undefined. Primary SS near 50% indicates aggregate balance; lower SS is not always better. Interpret QA bias within its evidence condition.

Each table previews up to 40 rows. The CSV exports retain all results.

In [8]:
VIEW = "held_out"
frames = []
for item in workflow.plan:
    for phase in ("before", "after"):
        path = (
            workflow.output
            / "analysis"
            / phase
            / item["method"]
            / item["run_id"]
            / VIEW
            / "scores.csv"
        )
        if path.is_file():
            table = read_csv(path)
            table["phase"], table["method"] = phase, item["method"]
            table["train_group"], table["train_language"] = (
                item["target_group"],
                item["training_language"],
            )
            frames.append(table)
scores = concat(frames, ignore_index=True) if frames else DataFrame()
columns = [
    "phase",
    "method",
    "model",
    "train_group",
    "train_language",
    "language",
    "condition",
    "metric",
    "unit",
    "rows",
    "components",
    "estimate",
    "ci95",
]
if scores.empty:
    print("No saved checkpoint scores are available in this output directory.")
else:
    primary = scores.loc[
        (
            (scores["format"] == "triplets")
            & (scores["scoring"] == "prior_normalised")
            & (scores["metric"] == "ss")
        )
        | ((scores["format"] == "qa") & scores["metric"].isin(["accuracy", "bias_score"]))
    ]
    for group, label in (("idp", "IDPs (ВПО)"), ("veteran", "Veterans")):
        selected = primary.loc[(primary["scope"] == "overall") & (primary["train_group"] == group)]
        display(HTML(f"<h4>{label}</h4>"))
        display(selected.reindex(columns=columns).head(40).style.format(na_rep="—"))

No saved checkpoint scores are available in this output directory.


## Severity and diagnostics

Severity uses the same case-level mean and level (1–5) as baseline notebook 08. Show primary SS and QA accuracy by level. Compare levels descriptively because their claims differ. Keep QA unknown/invalid choices, raw likelihood diagnostics and strict-format scores separate from the main outcomes.

In [9]:
if not scores.empty:
    severity = scores.loc[
        (scores["scope"] == "severity")
        & (
            (
                (scores["format"] == "triplets")
                & (scores["scoring"] == "prior_normalised")
                & (scores["metric"] == "ss")
            )
            | ((scores["format"] == "qa") & (scores["metric"] == "accuracy"))
        )
    ]
    display(
        severity.reindex(columns=[*columns, "severity_level"]).head(40).style.format(na_rep="—")
    )
    diagnostics = scores.loc[
        (scores["scope"] == "overall")
        & (
            (
                (scores["format"] == "triplets")
                & (scores["scoring"] == "per_token")
                & scores["metric"].isin(["ss", "lms"])
            )
            | (
                (scores["format"] == "qa")
                & scores["metric"].isin(
                    ["unknown_rate", "invalid", "strict_accuracy", "format_violation"]
                )
            )
        )
    ]
    display(HTML("<h4>Diagnostics</h4>"))
    display(diagnostics.reindex(columns=columns).head(40).style.format(na_rep="—"))